# European Option Pricing (Call & Put)
This notebook fetches 1 year of historical data using `yfinance`, computes the annualized historical volatility, and prices both European Call and Put options using the Black-Scholes model.

## Inputs

The Black-Scholes model prices a European option from **5 inputs**:

| Input | Symbol | Where we get it |
|---|---|---|
| Current stock price | $S_0$ | Market (yfinance, latest close) |
| Strike price | $K$ | Option contract |
| Time to maturity | $T$ | Expiry date - today (in years) |
| Risk-free rate | $r$ | Treasury bill yield (e.g. `^IRX`) |
| Volatility | $\sigma$ | Historical log returns of the stock |


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
import scipy.stats as stats
from scipy.optimize import brentq

# Fetch 1 year of historical daily data for TSLA
ticker_symbol = "^SPX"
df = yf.Ticker(ticker_symbol).history(period="1y")

# Extract the most recent closing price as the current stock price (S0)
S0 = df['Close'].iloc[-1]
print(f"Current {ticker_symbol} Stock Price (S0): ${S0:.2f}")

Current ^SPX Stock Price (S0): $7659.44


## Volatility from historical data

The model needs $\sigma$ = **annualized volatility** of the stock's log returns.

Daily log return:

$$r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)$$

Log returns add up over time, so their standard deviation grows with the square root of time. With 252 trading days in a year:

$$\boxed{\sigma_{\text{annual}} = \operatorname{std}(r_t)\times\sqrt{252}}$$

**Assumption:** log returns are (roughly) normally distributed with constant volatility - then the stock price is lognormal, which is exactly what Black-Scholes assumes.

In [2]:
# Calculate daily log returns: ln(P_t / P_t-1)
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Drop missing values if any
log_returns = df['Log_Return'].dropna()

# standard deviation and annualization 
daily_volatility = log_returns.std()
historical_volatility = daily_volatility * np.sqrt(252)

print(f"Daily Volatility: {daily_volatility:.6f}")
print(f"Annualized Historical Volatility (sigma): {historical_volatility:.4f} ({historical_volatility * 100:.2f}%)")

Daily Volatility: 0.008020
Annualized Historical Volatility (sigma): 0.1273 (12.73%)


## Definition of option parameters

## Option parameters and the $d_1$, $d_2$ terms

- **Strike** $K$ - price at which you can buy (call) / sell (put) the stock at expiry
- **Time to maturity** $T$ - years until expiry: $T = \dfrac{\text{days to expiry}}{365.25}$
- **Risk-free rate** $r$ - continuously compounded rate (here from the 13-week T-bill index `^IRX`)

The two quantities that appear everywhere in the formula:

$$d_1 = \frac{\ln(S_0/K) + \left(r + \tfrac{1}{2}\sigma^2\right)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

- $N(d_1)$ - how much of the stock you effectively "keep", chance-weighted for drift
- $N(d_2)$ - risk-neutral probability that the option expires in the money

In [3]:
# Define option parameters

K = 7640.0  # Strike Price
expiry_date = datetime(2026,11,13) 
today = datetime.now()

# ----------------------------------------------------------------------- #

#  time to maturity (T) in years
T = (expiry_date - today).days / 365.25
print(f"Time to Maturity (T): {T:.8f} years ({(expiry_date - today).days} days)")

# Risk-Free Rate from ^IRX and historical volatility:
irx = yf.Ticker("^IRX")
# recent close price
r = irx.history(period="5d")['Close'].iloc[-1] / 100 

sigma = historical_volatility
d1 = (np.log(S0 / K) + (r + 0.5 * (sigma ** 2)) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)

def black_scholes_call(S0, K, T, r, sigma):
    return S0 * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)

def black_scholes_put(S0, K, T, r, sigma):
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S0 * stats.norm.cdf(-d1)

Time to Maturity (T): 0.06570842 years (24 days)


## Option prices

## Black-Scholes pricing formulas

With $N(\cdot)$ = cumulative standard normal distribution:

**Call (right to buy):**
$$C = S_0\,N(d_1) - K e^{-rT} N(d_2)$$

**Put (right to sell):**
$$P = K e^{-rT} N(-d_2) - S_0 N(-d_1)$$

Reading the call formula: *stock part* - *cost part*. You receive stock worth $S_0 N(d_1)$ on average and pay $K e^{-rT} N(d_2)$ = the strike discounted to today.

**Put-call parity** gets its own section right below.

## Put-call parity (a free correctness check)

**In simple words:** a call and a put with the *same strike* and the *same expiry* are not independent - one can be built from the other plus some cash/stock, so their prices must stay in a fixed relationship.

**Why is it true?** Compare two portfolios that pay the **same amount at expiry**:

- **Portfolio A:** buy 1 call + keep $Ke^{-rT}$ cash in the bank. At expiry: if the stock is worth more than $K$ you exercise the call (you own the stock + the cash); if not, you just keep the cash. Either way you end up with $\max(S_T, K)$.
- **Portfolio B:** buy 1 put + buy 1 stock. At expiry: the put protects you if the stock falls below $K$, so you also end up with $\max(S_T, K)$.

Same payoff -> same price today:

$$C + Ke^{-rT} = P + S_0 \qquad\Longleftrightarrow\qquad \boxed{C - P = S_0 - Ke^{-rT}}$$

**Good to know:**

- Parity needs **no volatility and no model** - it follows from no-arbitrage alone, so it must hold *exactly*.
- If a real market broke parity, you could buy the cheap side and sell the rich side for a guaranteed profit (arbitrage).
- That is why the cell below prints `C - P` next to the *should be* value: a residual of ~0 proves our formulas are coded correctly.

In [4]:
# option prices
call_price = black_scholes_call(S0, K, T, r, sigma)
put_price = black_scholes_put(S0, K, T, r, sigma)

# put-call parity check:  C - P  should equal  S0 - K*e^(-rT)
parity_lhs = call_price - put_price
parity_rhs = S0 - K * np.exp(-r * T)

print("=" * 45)
print(f"Underlying Asset   : {ticker_symbol}")
print(f"Current Stock Price: ${S0:.2f}")
print(f"Strike Price       : ${K}")
print(f"Expiration Date and Time   : {expiry_date}    {T}")
print(f"Volatility Used    : {sigma * 100:.2f}%")
print("-" * 45)
print(f"Call Option Price  : ${call_price:.2f}")
print(f"Put Option Price   : ${put_price:.2f}")
print("-" * 45)
print(f"Put-call parity    : C - P = {parity_lhs:.4f}   (should be {parity_rhs:.4f})")
print(f"Parity residual    : {abs(parity_lhs - parity_rhs):.8f}")
print("=" * 45)

Underlying Asset   : ^SPX
Current Stock Price: $7659.44
Strike Price       : $7640.0
Expiration Date and Time   : 2026-11-13 00:00:00    0.06570841889117043
Volatility Used    : 12.73%
---------------------------------------------
Call Option Price  : $120.67
Put Option Price   : $80.79
---------------------------------------------
Put-call parity    : C - P = 39.8796   (should be 39.8796)
Parity residual    : 0.00000000


## Implied Volatility using Brent's root-finding method

## What is implied volatility?

**Normal way:** we know $\sigma$ -> compute the price.
**Implied volatility (IV):** we know the *market price* -> find the $\sigma$ that would produce it.

$$\text{find } \sigma \text{ such that } \underbrace{BS(S_0, K, T, r, \sigma)}_{\text{model price}} = \text{market price}$$

There is no algebraic formula to invert, so we solve it numerically with **Brent's method** (`scipy.optimize.brentq`): a safe root-finder applied to

$$f(\sigma) = BS(\sigma) - \text{market price} = 0$$

Useful properties: the model price always increases with $\sigma$, so the root is unique. If the market price is outside the model's allowed bounds (e.g. below intrinsic value), no solution exists -> it returns `NaN`.

In [5]:
def bs_price(CP, S0, K, sigma, T, r):
    """Calculates the Black-Scholes option price."""
    if sigma <= 0:
        return 0.0
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if CP.upper() == 'CALL':
        price = stats.norm.cdf(d1) * S0 - stats.norm.cdf(d2) * K * np.exp(-r * T)
    else:
        price = stats.norm.cdf(-d2) * K * np.exp(-r * T) - stats.norm.cdf(-d1) * S0
    return price

def implied_volatility(market_price, CP, S0, K, T, r):
    """
    Finds the implied volatility by reverse-engineering the Black-Scholes price 
    using Brent's root-finding method.
    """
    # Define an objective function: the difference between model price and market price
    def objective_function(sigma):
        return bs_price(CP, S0, K, sigma, T, r) - market_price

    try:
        # Search for volatility between 0.001 (0.1%) and 5.0 (500%)
        iv = brentq(objective_function, 1e-3, 5.0)
        return iv
    except ValueError:
        # Returns NaN if the market price is invalid
        return np.nan
    


S0 = 7659.44          # Current stock price
K = 7640           # Strike price
market_price_call = 160  # Observed market price of the call option
market_price_put = 90

iv_call = implied_volatility(market_price_call, 'CALL', S0, K, T, r)
iv_put =  implied_volatility(market_price_put, 'PUT', S0, K, T, r)

print()
    
print(f"Implied Volatility_CALL: {iv_call:.4f} ({iv_call * 100:.2f}%)")
print()
print(f"Implied Volatility_PUT: {iv_put:.4f} ({iv_put * 100:.2f}%)")
    


Implied Volatility_CALL: 0.1781 (17.81%)

Implied Volatility_PUT: 0.1392 (13.92%)
